In [2]:
# Nettoyage complet
print("Nettoyage...")
%reset -f

%whos

Nettoyage...
Interactive namespace is empty.


In [3]:
import numpy.random as rd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import pandas as pd 
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy import text
import sqlalchemy
import gc
from datetime import timedelta
import pyodbc
from datetime import datetime
from pandas import NaT
import math
from scipy.stats import expon
import geopandas as gpd
from shapely.geometry import Point
import contextily as ctx
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import time
from scipy.stats import gaussian_kde
import pickle

In [4]:
# Connexion pyodbc à SQL Server

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=172.16.11.33;"
    "DATABASE=LogiProDev;"
    "UID=quentin;"
    "PWD={Barbidur1;SQL}"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

requete_dates_transac = """ 
SELECT 
    tra_date_signature
FROM 
    t_transaction
WHERE
    tra_date_signature >= DATEADD(YEAR, -4, GETDATE())
"""

dates_transac = pd.read_sql_query(requete_dates_transac, conn)

conn.commit()
cursor.close()
conn.close()

C:\Users\quent\AppData\Local\Temp\ipykernel_23080\2559223756.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dates_transac = pd.read_sql_query(requete_dates_transac, conn)


In [5]:
# === 1️⃣ Préparer les intervalles entre transactions ===
dates_transac = dates_transac.sort_values('tra_date_signature')
dates = pd.to_datetime(dates_transac['tra_date_signature']).values
intervalles_jours = np.diff(dates) / np.timedelta64(1, 'D')  # en jours

# === 2️⃣ Estimer le paramètre lambda (intensité) ===
lambda_estime = 1 / np.mean(intervalles_jours)
print(f"λ estimé (transactions/jour) : {lambda_estime:.4f}")

# === 3️⃣ Sauvegarder le paramètre pour réutilisation ===
with open("poisson_transactions.pkl", "wb") as f:
    pickle.dump(lambda_estime, f)

print("✅ Processus de Poisson sauvegardé dans 'poisson_transactions.pkl'")

λ estimé (transactions/jour) : 14.7564
✅ Processus de Poisson sauvegardé dans 'poisson_transactions.pkl'
